# AgentBricks Knowledge Assistant SDK Support
AgentBricks Knowledge Assistant is now supported for creation via the Databricks Python SDK: https://databricks-sdk-py.readthedocs.io/en/latest/dbdataclasses/knowledgeassistants.html#databricks.sdk.service.knowledgeassistants.KnowledgeSourceState. In this notebook we quickly showcase how you can create a Knowledge Assistant Agent and also invoke it programmatically.

### Additional Resources/References
- https://github.com/databricks/databricks-sdk-py/blob/629374b64bc8d56e30b375800e9863f12bec7fc8/databricks/sdk/service/knowledgeassistants.py#L130
- https://www.databricks.com/blog/agent-bricks-knowledge-assistant-now-generally-available-turning-enterprise-knowledge-answers

## Setup
Upgrade the Databricks Python SDK to latest version to grab all API shapes.

In [0]:
!pip install --upgrade databricks-sdk

In [0]:
dbutils.library.restartPython()

## Configure Knowledge Assistant and Knowledge Source

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.knowledgeassistants import (
    FilesSpec,
    KnowledgeAssistant,
    KnowledgeAssistantState,
    KnowledgeSource,
    KnowledgeSourceState,
)
import time

print("Loading Databricks Upgraded SDK Modules")

In [0]:
w = WorkspaceClient()

uc_volume_path = "/Volumes/mock-data/rag-data/rag-volume"
for item in w.files.list_directory_contents(uc_volume_path):
    print(item.path, item.file_size, item.is_directory)

In [0]:
# ---------------------------------------------------------------------------
# Configuration – update these values before running
# ---------------------------------------------------------------------------
DISPLAY_NAME = "medium-article-assistant"
DESCRIPTION = "Answers questions about AWS and SageMaker specific Medium articles written by Ram Vegiraju. These also get very deep specifically about SageMaker Inference and the hosting of models as well."

In [0]:
# 1. Create the Knowledge Assistant
ka = w.knowledge_assistants.create_knowledge_assistant(
    KnowledgeAssistant(
        display_name=DISPLAY_NAME,
        description=DESCRIPTION,
        instructions="Answer concisely and cite your sources.",  # optional
        endpoint_name="ram-medium-ep-ka"
    )
)
print(f"Created Knowledge Assistant: {ka.name}  (id={ka.id})")

In [0]:
# 2. Add a knowledge source (files from a UC volume)
ks = w.knowledge_assistants.create_knowledge_source(
    parent=ka.name,  # e.g. "knowledge-assistants/<uuid>"
    knowledge_source=KnowledgeSource(
        display_name="internal-docs-ram",
        description="Medium articles around AWS and SageMaker",
        source_type="files",
        files=FilesSpec(path=uc_volume_path),
    ),
)
print(f"Added Knowledge Source: {ks.name}  (state={ks.state})")

In [0]:
POLL_INTERVAL_SECONDS = 120
def wait_for_knowledge_source(name: str) -> KnowledgeSource:
    """Poll until the Knowledge Source reaches UPDATED or FAILED_UPDATE."""
    while True:
        ks = w.knowledge_assistants.get_knowledge_source(name=name)
        print(f"  KnowledgeSource state: {ks.state}")
        if ks.state == KnowledgeSourceState.UPDATED:
            return ks
        if ks.state == KnowledgeSourceState.FAILED_UPDATE:
            raise RuntimeError(f"Knowledge Source failed to update: {name}")
        time.sleep(POLL_INTERVAL_SECONDS)
  
print("Waiting for Knowledge Source to finish indexing...")
ks = wait_for_knowledge_source(ks.name)
print(f"Knowledge Source is UPDATED. Ready to query: {ka.endpoint_name}")

## Sample Inference
Can invoke via the UI or using the API:

In [0]:
from openai import OpenAI
import os

# How to get your Databricks token: https://docs.databricks.com/en/dev-tools/auth/pat.html
DATABRICKS_TOKEN = os.getenv("DATABRICKS_TOKEN", "")
# Alternatively in a Databricks notebook you can use this:
# DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url="https://dbc-b1357123-778f.cloud.databricks.com/serving-endpoints"
)

response = client.responses.create(
    model=ka.endpoint_name,
    input=[
        {
            "role": "user",
            "content": "Who is Ram Vegiraju?"
        }
    ]
)

print(" ".join(getattr(content, "text", "") for output in response.output for content in getattr(output, "content", [])))

In [0]:
sample_payloads = ["Who is Ram Vegiraju?",
                   "What is the difference between Amazon SageMaker Multi-Model Endpoints and Multi-Container Endpoints?",
                   "What is SIP?",
                   "How do you enable AutoScaling with SageMaker Endpoints?"]

for payload in sample_payloads:
  response = client.responses.create(
    model=ka.endpoint_name,
    input=[
        {
            "role": "user",
            "content": payload
        }
    ]
  )

  print(" ".join(getattr(content, "text", "") for output in response.output for content in getattr(output, "content", [])))
  print("---------------------------------------")